# Chest X-ray classification walkthrough

This project builds a three-class image-classification benchmark around a public chest X-ray collection. The engineering work covers dataset contracts, exact-file integrity checks, duplicate-aware splitting, ResNet18 transfer learning, validation-based checkpoint selection and calibration-aware evaluation.

> The model numbers shown here come from an earlier run. They were not reproduced here. The earlier image-level split had five exact duplicate pairs identified afterwards.

In [1]:
import json
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "evidence" / "retained-results.json").is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from the project or one of its subdirectories")

spec = json.loads((ROOT / "data" / "dataset-spec.json").read_text())
retained_results = json.loads((ROOT / "evidence" / "retained-results.json").read_text())
duplicates = json.loads((ROOT / "evidence" / "known-exact-duplicates.json").read_text())

assert retained_results["reproduced_in_this_repository"] is False
print(f"Dataset contract: {spec['name']} v{spec['version']}")
print(f"Expected images: {spec['expected_total']}")
print(f"Result reproduced here: {retained_results['reproduced_in_this_repository']}")

Dataset contract: Chest X-Ray v1
Expected images: 3475
Result reproduced here: False


## Dataset composition

The repository expects 3,475 images across Normal, Lung Opacity and Viral Pneumonia. The images themselves are intentionally excluded. This figure comes from `data/dataset-spec.json`, so it is a dataset contract rather than a fresh inventory.

![Expected dataset composition](../assets/dataset-composition.svg)

In [2]:
total = spec["expected_total"]
print("class\texpected_count\tshare")
for label, item in spec["classes"].items():
    share = item["expected_count"] / total
    print(f"{label}\t{item['expected_count']}\t{share:.2%}")

class	expected_count	share
Normal	1250	35.97%
Lung Opacity	1125	32.37%
Viral Pneumonia	1100	31.65%


## Why exact duplicates matter

A random image split can put the same file, saved under two filenames, into both training and test data. The model then sees the test image during training and the reported score can look better than performance on genuinely unseen images.

The implementation uses SHA-256 as an exact identity key. Duplicate files are kept in one indivisible split group, while perceptual-hash matches remain review candidates rather than automatic grouping rules. Patient-level independence cannot be proved because patient identifiers are not supplied.

In [3]:
print(f"known exact duplicate pairs: {len(duplicates['duplicate_groups'])}")
print(f"class: {duplicates['class']}")
for pair in duplicates["duplicate_groups"]:
    print(f"{pair['members'][0]} <-> {pair['members'][1]}")

known exact duplicate pairs: 5
class: Viral Pneumonia
Pneumonia/1055.jpg <-> Pneumonia/134.jpg
Pneumonia/610.jpg <-> Pneumonia/991.jpg
Pneumonia/482.jpg <-> Pneumonia/963.jpg
Pneumonia/299.jpg <-> Pneumonia/402.jpg
Pneumonia/505.jpg <-> Pneumonia/994.jpg


## What the implementation demonstrates

| Area | Repository implementation |
| --- | --- |
| Data engineering | Public-source contract, class-count checks, readable-image checks and source-byte hashes |
| Leakage control | SHA-256 exact-identity groups, deterministic partitioning and cross-boundary audits |
| Deep learning | ResNet18 transfer learning with a three-class output layer and validation macro F1 selection |
| Evaluation | Accuracy, balanced accuracy, per-class precision/recall/F1, MCC, log loss, Brier score and calibration error |
| Reproducibility | Seeded runs, checkpoint binding, split/config digests and retained run metadata |

## Model results

The earlier run selected ResNet18 fine-tuning with dropout. It reported 0.9368 accuracy and 0.9381 macro F1 on 522 test images. These figures are retained for provenance, but they are not a current result from the safer exact-duplicate-grouped split.

![Model results](../assets/model-results.svg)

In [4]:
test = retained_results["test"]
print("Overall metrics")
for name in ("accuracy", "balanced_accuracy", "macro_f1", "matthews_correlation_coefficient"):
    print(f"{name}\t{test[name]:.4f}")
print("\nPer-class F1")
for label, values in test["per_class"].items():
    print(f"{label}\t{values['f1']:.4f}")

Overall metrics
accuracy	0.9368
balanced_accuracy	0.9376
macro_f1	0.9381
matthews_correlation_coefficient	0.9051

Per-class F1
Normal	0.9134
Lung Opacity	0.9102
Viral Pneumonia	0.9909


## What is available and what comes next

| Item | Status |
| --- | --- |
| Public dataset contract | Committed |
| Verification and duplicate-safe split code | Committed |
| Earlier-run metrics | Available; not reproduced here |
| Fresh grouped-split evaluation | Not available in this repository |
| Patient-level independence | Not established |
| External validation and calibration study | Not available |

To produce a current result, acquire the exact public dataset version named in `data/dataset-spec.json`, run `verify`, create the grouped split, train, and evaluate. Until that is done, the bars above must not be presented as leakage-safe performance. This is a research benchmark, not a medical device.